# Lab 4.5: Built-in Tools (Read, Write, Edit, Bash, Grep, Glob)

**What you'll build:** A decision engine that picks the right built-in tool for each codebase task -- and learn why the wrong tool wastes context and produces poor results.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- using Grep when you need Glob (and vice versa) | 5 min |
| 2 | The Right Way -- the built-in tool decision tree | 5 min |
| 3 | Your Turn -- classify 10 tasks to the correct built-in tool | 10 min |
| 4 | Edit Deep Dive -- handle unique matching and replace_all | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

Claude Code has 6 built-in tools. Each has a specific purpose:

| Tool | Purpose | Key Detail |
|------|---------|------------|
| **Grep** | Search FILE CONTENT | Regex patterns, returns matching lines |
| **Glob** | Search FILE PATHS | Glob patterns like `**/*.py`, returns file paths |
| **Read** | Read a known file | Full file or line range, requires knowing the path |
| **Edit** | Replace text in a file | old_string must be UNIQUE in the file |
| **Write** | Create or overwrite a file | Full file replacement, prefer Edit for partial changes |
| **Bash** | Execute shell commands | For running scripts, installing packages, etc. |

The critical distinction: **Grep = content search, Glob = path search**.

---

## Phase 1: The Wrong Approach

Let's see what happens when you use the wrong tool for a task.

In [ ]:
# Demonstration: Grep vs Glob confusion

WRONG_TOOL_EXAMPLES = [
    {
        "task": "Find all Python test files in the project",
        "wrong_tool": "Grep",
        "wrong_reason": "Grep searches file CONTENT, not file NAMES. Searching content for 'test_' might match variable names, comments, or imports -- not just test files.",
        "right_tool": "Glob",
        "right_usage": 'Glob(pattern="**/test_*.py") -- searches file PATHS, returns only test files.',
    },
    {
        "task": "Find where the function 'process_order' is defined",
        "wrong_tool": "Glob",
        "wrong_reason": "Glob searches file NAMES. 'process_order' is a function, not a file -- Glob won't find it.",
        "right_tool": "Grep",
        "right_usage": 'Grep(pattern="def process_order") -- searches file CONTENT for the definition.',
    },
    {
        "task": "Change 'API_VERSION = 2' to 'API_VERSION = 3' in config.py",
        "wrong_tool": "Write",
        "wrong_reason": "Write replaces the ENTIRE file. For a single-line change, this wastes tokens transferring all the unchanged content.",
        "right_tool": "Edit",
        "right_usage": 'Edit(old_string="API_VERSION = 2", new_string="API_VERSION = 3") -- targeted replacement.',
    },
    {
        "task": "Understand the structure of an unfamiliar project",
        "wrong_tool": "Read (reading every file)",
        "wrong_reason": "Reading every file wastes context tokens on irrelevant content. Most files in a project are not needed for structural understanding.",
        "right_tool": "Glob then Read",
        "right_usage": 'Glob to find key files (README, package.json, main entry points), then Read those specific files.',
    },
]

print("=== WRONG TOOL EXAMPLES ===\n")
for ex in WRONG_TOOL_EXAMPLES:
    print(f"Task: {ex['task']}")
    print(f"  WRONG: {ex['wrong_tool']} -- {ex['wrong_reason']}")
    print(f"  RIGHT: {ex['right_tool']} -- {ex['right_usage']}")
    print()

---

## Phase 2: The Right Approach

A decision tree for selecting the right built-in tool.

In [ ]:
def select_builtin_tool(task_description: str) -> dict:
    """Use Claude to select the right built-in tool for a task."""
    prompt = f"""You are a Claude Code assistant. Given a codebase task, select the correct built-in tool.

Available tools:
- Grep: Search FILE CONTENT using regex patterns. Returns matching lines.
- Glob: Search FILE PATHS using glob patterns (e.g., **/*.py). Returns file paths.
- Read: Read the contents of a KNOWN file (you must know the path).
- Edit: Replace UNIQUE text in a file (old_string must appear exactly once, or use replace_all).
- Write: Create a new file or completely overwrite an existing file.
- Bash: Execute shell commands (run scripts, install packages, etc.).

Decision rules:
- Need to find files by name/extension? -> Glob
- Need to find text inside files? -> Grep
- Need to read a specific file you already know? -> Read
- Need to change part of a file? -> Edit
- Need to create a new file or rewrite entirely? -> Write
- Need to run a command or script? -> Bash

Task: {task_description}

Return ONLY a JSON object with:
- "tool": the tool name (Grep, Glob, Read, Edit, Write, or Bash)
- "reason": one sentence explaining why
- "usage": example of how you'd use it
"""
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"tool": "unknown", "reason": raw, "usage": ""}


# Test the decision tree
DEMO_TASKS = [
    ("Find all YAML config files in the project", "Glob"),
    ("Find all uses of the deprecated 'old_api' function", "Grep"),
    ("Read the package.json to check dependencies", "Read"),
    ("Change the port number from 8080 to 9090 in server.py", "Edit"),
    ("Create a new Dockerfile for the project", "Write"),
    ("Run the test suite with pytest", "Bash"),
]

print("=== BUILT-IN TOOL DECISION TREE ===\n")
for task, expected in DEMO_TASKS:
    result = select_builtin_tool(task)
    selected = result.get("tool", "unknown")
    correct = selected == expected
    tag = "OK" if correct else "XX"
    print(f"  [{tag}] Task: {task}")
    print(f"       Tool: {selected} (expected: {expected})")
    print(f"       Reason: {result.get('reason', 'N/A')}")
    print()

---

## Phase 3: Your Turn

Classify 10 codebase tasks to the correct built-in tool.

**Your goal:** 100% accuracy on tool selection.

In [ ]:
CHALLENGE_TASKS = [
    {"task": "Find all Markdown files in the docs/ directory", "answer": "Glob"},
    {"task": "Find which file contains the class 'DatabaseConnection'", "answer": "Grep"},
    {"task": "Read the README.md to understand the project", "answer": "Read"},
    {"task": "Rename the variable 'max_retry' to 'max_retries' in utils.py (appears 8 times)", "answer": "Edit"},
    {"task": "Create a new .gitignore file", "answer": "Write"},
    {"task": "Install the numpy package", "answer": "Bash"},
    {"task": "Find all TODO comments across the codebase", "answer": "Grep"},
    {"task": "List all JSON configuration files in the project", "answer": "Glob"},
    {"task": "Change the API endpoint URL in one specific line of config.py", "answer": "Edit"},
    {"task": "Check if the Docker daemon is running", "answer": "Bash"},
]

print("Challenge: Select the correct built-in tool for each task.")
print("Fill in YOUR_ANSWERS below.\n")

In [ ]:
# =============================================================
# TODO: Fill in the correct tool for each task
# =============================================================
#
# Options: "Grep", "Glob", "Read", "Edit", "Write", "Bash"
#
# Think about:
# - Am I searching content or paths?
# - Am I reading, modifying, or creating?
# - Am I running a command?

YOUR_ANSWERS = [
    "",  # Task 1: Find all Markdown files in docs/
    "",  # Task 2: Find which file contains 'DatabaseConnection'
    "",  # Task 3: Read the README.md
    "",  # Task 4: Rename variable in utils.py (8 occurrences)
    "",  # Task 5: Create a new .gitignore
    "",  # Task 6: Install numpy
    "",  # Task 7: Find TODO comments
    "",  # Task 8: List JSON config files
    "",  # Task 9: Change API endpoint URL in one line
    "",  # Task 10: Check Docker daemon
]

# Check your answers
print("=== YOUR SCORECARD ===\n")
correct = 0
for i, (task_info, answer) in enumerate(zip(CHALLENGE_TASKS, YOUR_ANSWERS), 1):
    expected = task_info["answer"]
    is_correct = answer == expected
    if is_correct:
        correct += 1
    tag = "PASS" if is_correct else "FAIL"
    print(f"  [{tag}] Task {i}: {task_info['task'][:50]}...")
    if not is_correct:
        print(f"         Your answer: {answer or '(empty)'} | Correct: {expected}")

print(f"\nScore: {correct}/{len(CHALLENGE_TASKS)}")
if correct == len(CHALLENGE_TASKS):
    print("PASSED -- you correctly selected the built-in tool for every task!")
else:
    print("Review the decision tree: Grep=content, Glob=paths, Edit=modify, Write=create, Bash=execute")

---

## Phase 4: Edit Deep Dive

The Edit tool has a critical constraint: `old_string` must be unique in the file. Let's explore this and the `replace_all` escape hatch.

In [ ]:
# Simulating Edit behavior
SAMPLE_FILE = """import os
import sys

def process_data(data):
    result = []
    for item in data:
        result.append(item.strip())
    return result

def process_batch(batch):
    results = []
    for item in batch:
        results.append(process_data([item]))
    return results

# Process the final output
output = process_data(input_data)
"""


def simulate_edit(file_content, old_string, new_string, replace_all=False):
    """Simulate Edit tool behavior."""
    count = file_content.count(old_string)
    
    if count == 0:
        return {"status": "error", "message": f"old_string not found in file"}
    elif count > 1 and not replace_all:
        return {
            "status": "error",
            "message": f"old_string appears {count} times -- must be unique. "
                       f"Add more context or use replace_all=True."
        }
    else:
        if replace_all:
            new_content = file_content.replace(old_string, new_string)
            return {"status": "success", "replacements": count, "content": new_content}
        else:
            new_content = file_content.replace(old_string, new_string, 1)
            return {"status": "success", "replacements": 1, "content": new_content}


# Case 1: Unique match (works)
print("=== CASE 1: Unique match (works) ===")
result = simulate_edit(SAMPLE_FILE, "def process_data(data):", "def transform_data(data):")
print(f"  Status: {result['status']}")
print(f"  Replacements: {result.get('replacements', 0)}")

# Case 2: Non-unique match (fails)
print("\n=== CASE 2: Non-unique match (fails) ===")
result = simulate_edit(SAMPLE_FILE, "result", "output")
print(f"  Status: {result['status']}")
print(f"  Message: {result.get('message', '')}")

# Case 3: More context makes it unique (works)
print("\n=== CASE 3: More context makes it unique (works) ===")
result = simulate_edit(SAMPLE_FILE, "    result = []\n    for item in data:", "    result = []\n    for item in data:  # Process each item")
print(f"  Status: {result['status']}")
print(f"  Replacements: {result.get('replacements', 0)}")

# Case 4: replace_all for global rename (works)
print("\n=== CASE 4: replace_all for global rename (works) ===")
result = simulate_edit(SAMPLE_FILE, "process_data", "transform_data", replace_all=True)
print(f"  Status: {result['status']}")
print(f"  Replacements: {result.get('replacements', 0)}")
print(f"  All 'process_data' instances renamed to 'transform_data'.")

### Key Takeaways

1. **Grep = content search, Glob = path search.** This is the #1 distinction to remember.
2. **Incremental exploration: Glob -> Read -> Grep.** Start with file structure, read key files, then search for specifics.
3. **Edit requires unique old_string.** If not unique, add surrounding context or use replace_all.
4. **Prefer Edit over Write for modifications.** Edit sends only the diff; Write sends the entire file.
5. **Bash is for execution.** Running scripts, installing packages, checking system state -- anything that requires a shell.
6. **The wrong tool wastes tokens.** Using Grep to find files (instead of Glob) returns irrelevant content matches. Using Read on every file (instead of Grep) transfers unnecessary content.